# Stage 7 development retrieval core
Tesla T4, Internet and GITHUB_TOKEN. This notebook runs the implemented development subset.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import tempfile
import uuid

remote = 'https://github.com/ArcheLabs/mini-cells.git'
branch = 'research/minicells-08-runtime'
run_id = 'stage7-development-' + uuid.uuid4().hex[:12]
output = Path('/kaggle/working') / run_id
output.mkdir()
with tempfile.TemporaryDirectory() as tmp:
    repo = Path(tmp) / 'repo'
    subprocess.run(['git', 'clone', '--depth=1', '--branch', branch, remote, str(repo)],
                   check=True, timeout=300)
    source = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()
    sys.path.insert(0, str(repo))
    from scripts.research.publish_experiment_results import (
        ExperimentSpec, load_github_token, prepare_artifacts, push_results,
    )
    load_github_token('GITHUB_TOKEN')
    log = output / 'raw.log'
    try:
        with log.open('w') as stream:
            subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers==5.3.0'],
                           stdout=stream, stderr=subprocess.STDOUT, check=True, timeout=600)
            subprocess.run([sys.executable, '-u', str(repo / 'research/titans/stage7/run_stage7.py'),
                            '--output', str(output)], cwd=repo,
                           check=True, timeout=10200)
        decision = json.loads((output / 'smoke.json').read_text())
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
        print(log.read_text(errors='replace')[-6000:], flush=True)
        decision = {'status': 'ERROR', 'error': str(exc)}
    result_branch = 'kaggle/' + run_id
    decision.update(format='minicells.stage7.development.v1', source_commit=source, run_id=run_id)
    (output / 'decision.json').write_text(json.dumps(decision, indent=2))
    stage = repo / 'results/stage7-development'
    stage.mkdir(parents=True)
    names = ('decision.json', 'raw.log') + tuple(p.name for p in output.iterdir() if p.suffix in ('.json', '.md') and p.name not in ('decision.json', 'publication.json'))
    for name in names:
        shutil.copy2(output / name, stage / name)
    spec = ExperimentSpec('results/stage7-development', 'artifacts/experiments/minicells08/' + run_id,
                          names, result_branch, decision['format'])
    dest = prepare_artifacts(repo, 'minicells08', spec, None)
    push_results(repo, dest, 'minicells08', result_branch, 'GITHUB_TOKEN', 'origin/main')
    receipt = {'status': 'REMOTE_SHA_VERIFIED', 'branch': result_branch,
               'commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()}
    (output / 'publication.json').write_text(json.dumps(receipt, indent=2))
    print(decision, receipt, flush=True)
    if decision['status'] != 'MINICELLS_MVP_INCOMPLETE':
        raise RuntimeError(f'Inspect {log}')
